In [1]:
import sqlite3 as sql
import pandas as pd
import sqlalchemy

forzar fecha desde un inicio para no tener problemas mas adelante

In [2]:
df=pd.read_csv("ventas_techventas.csv", encoding="latin1")
dfCopy=df.copy()
dfCopy["fecha"] = pd.to_datetime(dfCopy["fecha"], errors="coerce")

dfCopy["anio"]          = dfCopy["fecha"].dt.year
dfCopy["mes"]           = dfCopy["fecha"].dt.month
dfCopy["dia"]           = dfCopy["fecha"].dt.day
dfCopy["nombre_mes"]    = dfCopy["fecha"].dt.month_name()
dfCopy["dia_semana"]    = dfCopy["fecha"].dt.day_name()
dfCopy["trimestre"]     = dfCopy["fecha"].dt.quarter

dfCopy[["fecha", "anio", "mes", "nombre_mes", "dia", "dia_semana", "trimestre"]].head(8)



,fecha,anio,mes,nombre_mes,dia,dia_semana,trimestre
0,2024-01-05,2024.0,1.0,January,5.0,Friday,1.0
1,2024-01-07,2024.0,1.0,January,7.0,Sunday,1.0
2,NaT,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-01-12,2024.0,1.0,January,12.0,Friday,1.0
4,2024-01-15,2024.0,1.0,January,15.0,Monday,1.0
5,2024-01-18,2024.0,1.0,January,18.0,Thursday,1.0
6,NaT,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-01-22,2024.0,1.0,January,22.0,Monday,1.0


Quitamos los datos nulos

In [3]:
dfSinNan= dfCopy.dropna()
dfSinNan.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor,anio,mes,dia,nombre_mes,dia_semana,trimestre
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.10,Ana Lï¿½ï¿,2024.0,1.0,5.0,January,Friday,1.0
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz,2024.0,1.0,7.0,January,Sunday,1.0
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora,2024.0,1.0,12.0,January,Friday,1.0
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3.0,1200.0,0.15,Carlos Ruiz,2024.0,1.0,15.0,January,Monday,1.0
5,1006,2024-01-18,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,20.0,95.0,0.00,Maria Torres,2024.0,1.0,18.0,January,Thursday,1.0


Hacemos un replace por error de codificacion en la base

In [4]:
dfSinNan['vendedor'] = dfSinNan['vendedor'].str.replace('.*Ana.*', 'Ana Lopez', regex=True)
dfSinNan.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor,anio,mes,dia,nombre_mes,dia_semana,trimestre
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.10,Ana Lopez,2024.0,1.0,5.0,January,Friday,1.0
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz,2024.0,1.0,7.0,January,Sunday,1.0
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora,2024.0,1.0,12.0,January,Friday,1.0
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3.0,1200.0,0.15,Carlos Ruiz,2024.0,1.0,15.0,January,Monday,1.0
5,1006,2024-01-18,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,20.0,95.0,0.00,Maria Torres,2024.0,1.0,18.0,January,Thursday,1.0


# Consultas SQL 
conectamos con "servidor" en memoria e importamos el DataFrame a SQL
y realizamos la coneccion

In [5]:

conn=sql.connect(":memory:")

dfSinNan.to_sql("ventas_techventas", conn, if_exists="replace", index=False )

tablas=pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tabla cargada: ")
print(tablas.to_string(index=False))

Tabla cargada: 
             name
ventas_techventas


# Productos unicos

In [6]:
# Primer consulta, productos unicos con alias descriptivo
q1 = pd.read_sql("""
    -- seleccionamos el los unicos de las columnas producto y cantidad
    -- (DISTINCT)
    SELECT DISTINCT producto AS productoUnico, 
    SUM(cantidad)
    
    FROM ventas_techventas
    
    WHERE producto IS NOT NULL 
    GROUP BY producto
    ORDER BY cantidad DESC
    
    LIMIT 5
""", conn)
q1

,productoUnico,SUM(cantidad)
0,SSD 1TB,260.0
1,Mouse Inalambrico,155.0
2,Teclado Mecanico,60.0
3,Router WiFi 6,39.0
4,Laptop Pro 15,29.0


# Filtrar pedidos del primer trimestre mayor a 5 unidades

In [7]:
#Filtra pedidos del primer trimestre (ene–mar 2024) con cantidad
#mayor a 5 unidades.

q2=pd.read_sql("""
    SELECT fecha, cliente_id as id, cliente_nombre as nombre, producto, cantidad ,precio_unitario as precio, mes
    FROM ventas_techventas
    WHERE trimestre == 1 AND cantidad > 5
    
""", conn)
q2.head(8)

,fecha,id,nombre,producto,cantidad,precio,mes
0,2024-01-07 00:00:00,C002,Innovatek,Mouse Inalambrico,15.0,25.0,1.0
1,2024-01-12 00:00:00,C001,TechCorp SA,Teclado Mecanico,10.0,85.0,1.0
2,2024-01-18 00:00:00,C005,Sistemas Globales,SSD 1TB,20.0,95.0,1.0
3,2024-01-22 00:00:00,C006,NetPrime,Router WiFi 6,8.0,120.0,1.0
4,2024-01-25 00:00:00,C003,DataSolutions,SSD 1TB,12.0,95.0,1.0
5,2024-02-05 00:00:00,C008,BetaSoft,Mouse Inalambrico,30.0,25.0,2.0
6,2024-02-12 00:00:00,C005,Sistemas Globales,SSD 1TB,25.0,95.0,2.0
7,2024-02-20 00:00:00,C002,Innovatek,SSD 1TB,18.0,95.0,2.0


# Vendedores con ingreso mayor a $10.000

In [8]:
#Vendedores cuyo ingreso bruto total (cantidad × precio) supera
#$10,000 USD.
q3=pd.read_sql("""
    SELECT 
        vendedor,
        
        -- SUM es la funcion para hacer la operacion
        SUM(cantidad*precio_unitario)       AS Ingreso
    
    FROM ventas_techventas
    
    -- Agrupamos con el indice de vendedor 
    GROUP BY vendedor
    
    --definimos el filtro
    HAVING Ingreso > 10000
    
""", conn)
q3

,vendedor,Ingreso
0,Ana Lopez,20810.0
1,Carlos Ruiz,21355.0
2,Luis Mora,19830.0
3,Maria Torres,11160.0


In [9]:
#COUNT + SUM + AVG
#Por categoría: total de pedidos, suma de unidades vendidas y
#precio unitario promedio.

q4= pd.read_sql("""
    SELECT 
        categoria,
        COUNT(*)                        as cantidad,
        SUM(cantidad)                   as Unidades_vendidas,
        AVG(precio_unitario)            as promedio_de_precio
    FROM ventas_techventas
    GROUP BY categoria
""", conn)
q4

,categoria,cantidad,Unidades_vendidas,promedio_de_precio
0,Almacenamiento,12,260.0,95.0
1,Electronica,13,29.0,1200.0
2,Perifericos,15,215.0,53.0
3,Redes,8,39.0,120.0


# Creacion de Tabla Vendedores 

In [10]:
#Crea la tabla vendedores (abajo) y calcula el % de
#cumplimiento de meta de cada uno.
conn.executescript("""
    CREATE TABLE Tvendedores(
    name_vendedor        TEXT,
    name_zona            TEXT,
    meta_mensual    INTEGER
);
""")
print("tabla Vendedores creada")

tabla Vendedores creada


insertar valores

In [11]:
conn.executescript("""

    INSERT INTO Tvendedores Values
    ('Ana Lopez','Norte', 8000),
    ('Carlos Ruiz', 'Sur', 7500),
    ('Luis Mora', 'Norte', 8000),
    ('Maria Torres', 'Centro', 7000);
""")

In [ ]:
tabla= pd.read_sql("""
    SELECT * FROM Tvendedores
""", conn)
tabla

,name_vendedor,name_zona,meta_mensual
0,Ana Lopez,Norte,8000
1,Carlos Ruiz,Sur,7500
2,Luis Mora,Norte,8000
3,Maria Torres,Centro,7000


# Calculo % cumplimiento de cada vendedor 
 

In [ ]:
q5 = pd.read_sql("""
    SELECT DISTINCT
        name_vendedor,
        name_zona,
        meta_mensual,
        ROUND(SUM(cantidad * precio_unitario)) AS ventas_totales,
        ROUND((SUM(cantidad * precio_unitario) / meta_mensual) * 100) AS porcentaje_cumplimiento
    FROM Tvendedores
    LEFT JOIN ventas_techventas ON vendedor = name_vendedor
    GROUP BY name_vendedor, name_zona, meta_mensual
""", conn)
q5

,name_vendedor,name_zona,meta_mensual,ventas_totales,porcentaje_cumplimiento
0,Ana Lopez,Norte,8000,20810.0,260.0
1,Carlos Ruiz,Sur,7500,21355.0,285.0
2,Luis Mora,Norte,8000,19830.0,248.0
3,Maria Torres,Centro,7000,11160.0,159.0


# Cliente con mayor ingreso en el primer semestre

In [14]:
#Encuentra el cliente que generó el mayor ingreso total en el
#primer semestre.

subconsulta=pd.read_sql("""
    SELECT cliente_nombre AS cliente,
        ROUND(SUM(descuento*precio_unitario*cantidad))  AS Ingreso
    FROM ventas_techventas
    WHERE mes <= 6
    GROUP BY cliente
    ORDER BY Ingreso DESC
    

""", conn)
subconsulta

,cliente,Ingreso
0,TechCorp SA,3038.0
1,CloudNet,1210.0
2,BetaSoft,820.0
3,DeltaOps,480.0
4,Sistemas Globales,478.0
5,DataSolutions,384.0
6,GammaDevs,209.0
7,AlphaTech,110.0
8,NetPrime,102.0
9,Innovatek,24.0
